In [ ]:
import h5py
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# DATASET FILES

Dataset_files = [
    "H-H1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_16KHZ_R1-1187008867-32.hdf5"
    "H-H1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187006835-4096.hdf5"
]

# LOAD STRAIN DATA

def load_strain(file_path):
    with h5py.File(file_path, "r") as f:
        strain = f["strain"]["Strain"][:]
    return strain

# CREATE WINDOWS

WINDOW_SIZE = 4096
STEP_SIZE = 2048

X = []
y = []

def create_windows(signal, label):
    for i in range(0, len(signal) - WINDOW_SIZE, STEP_SIZE):
        window = signal[i:i + WINDOW_SIZE]
        X.append(window)
        y.append(label)

# Positive samples
for file in positive_files:
    signal = load_strain(file)
    create_windows(signal, 1)

# Negative samples
for file in negative_files:
    signal = load_strain(file)
    create_windows(signal, 0)

X = np.array(X)
y = np.array(y)

print("Dataset Shape:", X.shape)
print("Labels Shape:", y.shape)

# NORMALIZATION

scaler = StandardScaler()

X = scaler.fit_transform(X)

# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ANN MODEL

model = Sequential()

model.add(Dense(
    512,
    activation='relu',
    input_shape=(X_train.shape[1],)
))

model.add(Dropout(0.3))

model.add(Dense(
    256,
    activation='relu'
))

model.add(Dropout(0.3))

model.add(Dense(
    128,
    activation='relu'
))

model.add(Dropout(0.2))

model.add(Dense(
    64,
    activation='relu'
))

model.add(Dense(
    1,
    activation='sigmoid'
))


# COMPILE MODEL


model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#EARLY STOPPING


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# TRAIN MODEL

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# EVALUATE MODEL

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", accuracy)

# PREDICTIONS

predictions = model.predict(X_test)

predictions = (
    predictions > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        predictions
    )
)

# SAVE MODEL
model.save("ANN_NSBH_Detector.h5")

print("Model Saved")